# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is described via a Croissant schema at the URL below.

> **URL:** https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Show high-level metadata information
meta = dataset.metadata
print(f"Name: {getattr(meta, 'name', None)}\nDescription: {getattr(meta, 'description', None)}")
print(f"Authors: {getattr(meta, 'author', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")

## 2. Data Overview
Review available record sets, fields, columns, and their IDs.

We'll enumerate the dataset's record sets, displaying their `@id` and `name`, with their field details and field/column IDs. This helps you select which record sets and fields to analyze.

In [ ]:
# List available record sets and their fields/columns
record_sets = getattr(meta, 'recordSets', None)
if record_sets is None:
    # Try 'recordSet' key instead (older Croissant schemas)
    record_sets = getattr(meta, 'recordSet', None)

all_record_set_ids = []
if record_sets:
    for record_set in record_sets:
        rs_id = getattr(record_set, '@id', None)
        rs_name = getattr(record_set, 'name', None)
        all_record_set_ids.append(rs_id)
        print(f"RecordSet: {rs_name} (@id={rs_id})")
        # Print field details
        fields = getattr(record_set, 'fields', None)
        if fields:
            for field in fields:
                field_id = getattr(field, '@id', None)
                field_name = getattr(field, 'name', None)
                print(f"  Field: {field_name} (@id={field_id})")
                columns = getattr(field, 'columns', None)
                if columns:
                    for col in columns:
                        col_id = getattr(col, '@id', None)
                        col_name = getattr(col, 'name', None)
                        print(f"    Column: {col_name} (@id={col_id})")
        print()
else:
    print("No record sets found in metadata. The dataset schema may require inspecting available records directly.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Specify the desired record set `@id` to extract the corresponding records as a DataFrame. We'll attempt to extract all available top-level record sets as listed above.

> **Tip:** If no record set is found above, you can obtain all available IDs using `dataset.record_set_ids()`.

In [ ]:
# If the earlier cell didn't enumerate record sets, use dataset helper
if not all_record_set_ids:
    try:
        all_record_set_ids = dataset.record_set_ids()
        print("Record Set IDs:", all_record_set_ids)
    except Exception as e:
        print("Error enumerating record set ids:", e)

# For the FAIR^2 dataset, let's try all available record sets
if not all_record_set_ids:
    print("No record set IDs found. Check dataset schema or documentation.")
else:
    dataframes = {}
    for record_set_id in all_record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            if records:  # Not empty
                df = pd.DataFrame(records)
                dataframes[record_set_id] = df
                print(f"RecordSet '@id': {record_set_id} - Loaded {len(df)} records, columns: {df.columns.tolist()}")
            else:
                print(f"RecordSet '@id': {record_set_id} - No records.")
        except Exception as ex:
            print(f"Could not extract records for {record_set_id}: {ex}")

# Show sample records (using the first available)
if dataframes:
    some_rs_id = list(dataframes.keys())[0]
    print(f"\nSample records from RecordSet {some_rs_id}:")
    display(dataframes[some_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply standard data processing and exploration. Here, we'll select a numeric field from the extracted DataFrame and perform basic filtering, normalization, and grouping. **All fields and columns must be referenced by their `@id`.**

Make sure to update the field IDs below to match the ones discovered in your dataset.

In [ ]:
# Pick any DataFrame with numeric fields (inspect columns to determine which ones are numeric)
target_record_set_id = None
for rsid, df in dataframes.items():
    numeric_cols = df.select_dtypes(include=['float64', 'int64']).columns.tolist()
    if numeric_cols:
        target_record_set_id = rsid
        break

if target_record_set_id is None:
    print("No numeric columns found in loaded record sets.")
else:
    df = dataframes[target_record_set_id]
    print(f"Chosen RecordSet '@id': {target_record_set_id}")
    print("Numeric fields detected:", df.select_dtypes(include=['float64', 'int64']).columns.tolist())

    # Select a numeric field (by column `@id`/column name)
    # Update this string to one of the detected numeric fields (use actual column name as imported)
    numeric_field = df.select_dtypes(include=['float64', 'int64']).columns[0]

    # Example: filter for values above a threshold
    threshold = df[numeric_field].mean()  # E.g., above mean
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by a categorical field
    groupable_fields = df.select_dtypes(include=['object']).columns.tolist()
    if groupable_fields:
        group_field = groupable_fields[0]
        grouped_df = filtered_df.groupby(group_field, as_index=False)[numeric_field].mean()
        print(f"\nGrouped data by {group_field}:")
        display(grouped_df.head())

## 5. Visualization
Visualize the distribution of the selected numeric field and (optionally) its grouping.

We'll plot histograms and (if available) grouped means by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if target_record_set_id is not None:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    # If grouped_df from above exists
    if 'grouped_df' in locals():
        plt.figure(figsize=(10, 5))
        sns.barplot(x=group_field, y=numeric_field, data=grouped_df)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR^2 mass spectrometry analysis dataset using the `mlcroissant` library, loaded metadata and tabular data by referencing entities with their `@id`s, and demonstrated filtering, normalization, and visualization of numeric records. For deeper biological or domain-specific analyses, consult the dataset documentation and refer to field descriptions using their `@id`.